목표:
- 평가 문서 30개 중 1개 문서를 대상으로 per-question 생성 답변을 평가
- 질문별로 gold/pred 및 지표(ret_recall, ret_mrr, gen_fill, gen_match, gen_sim)를 확인
- 문서 전체 평균도 함께 표시
- 결과는 json 1개로 저장 (doc index/id + 질문별 상세 + 문서 평균)

참고 사항:
- 기존 코드의 per-question 실험 구조와 추상화 클래스(RAGExperiment 등)를 그대로 사용
- retrieval 상세(ctx/chunk idx 등)는 저장/표시하지 않음
- gen_match/gen_sim은 evalgenpred()의 rapidfuzz token_set_ratio 기반(기존 구현 그대로)

In [1]:
import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer

from preprocess.pp_basic import docs, BASE_DIR, GOLD_EVIDENCE_CSV, GOLD_FIELDS_JSONL
from preprocess.rag_experiment_per_question import (
    CONFIG,
    ExperimentSpec,
    load_questions_df,
    make_components,
    RAGExperiment,
)

d:\dev\github\codeit-part3-team4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv(), override=False)

client = OpenAI()
embedmodel = SentenceTransformer("nlpai-lab/KoE5")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 737.10it/s, Materializing param=pooler.dense.weight]                               


### Load gold (CSV + JSONL) + questions

In [3]:
gold_evidence_df = pd.read_csv(GOLD_EVIDENCE_CSV)

def load_gold_fields_jsonl(path: Path) -> pd.DataFrame:
    out = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)

            iid = r.get("instance_id")
            doc_id = r.get("doc_id")  # JSONL top-level key
            fields = r.get("fields", {}) or {}

            for k, v in fields.items():
                out.append({"instance_id": iid, "doc_id": doc_id, "field": k, "gold": v})

    return pd.DataFrame(out)

gold_fields_df = load_gold_fields_jsonl(Path(GOLD_FIELDS_JSONL))
questions_df = load_questions_df()

print("gold_evidence_df:", gold_evidence_df.shape, "cols:", gold_evidence_df.columns.tolist())
print("gold_fields_df:", gold_fields_df.shape, "cols:", gold_fields_df.columns.tolist())
print("gold_fields_df unique doc_id:", gold_fields_df["doc_id"].astype(str).nunique())
print("questions_df:", questions_df.shape, "cols:", questions_df.columns.tolist())
print("ndocs(all):", len(docs))

gold_evidence_df: (420, 5) cols: ['instance_id', 'doc_id', 'page_start', 'page_end', 'anchor_text']
gold_fields_df: (420, 4) cols: ['instance_id', 'doc_id', 'field', 'gold']
gold_fields_df unique doc_id: 20
questions_df: (311, 5) cols: ['instance_id', 'qid', 'doc_id', 'question', 'type']
ndocs(all): 100


### Build EVAL_DOCS + choose target by index

In [4]:
def namekey(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s)).strip()
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s)
    return s

gold_docid_set = set(namekey(x) for x in gold_fields_df["doc_id"].astype(str).dropna().unique())
EVAL_DOCS = [p for p in docs if namekey(p.name) in gold_docid_set]

print("Eval docs:", len(EVAL_DOCS))
print("Eval doc_id examples:", [p.name for p in EVAL_DOCS[:5]])

if not EVAL_DOCS:
    raise RuntimeError("EVAL_DOCS is empty. doc_id matching failed.")

Eval docs: 20
Eval doc_id examples: ['(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf', '(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.pdf', '(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.pdf', '(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.pdf', '2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.pdf']


In [5]:
# EVAL_DOCS 전체 문서명 확인 (0-based 인덱스 포함)
eval_doc_names = [p.name for p in EVAL_DOCS]

print(f"Total EVAL_DOCS: {len(eval_doc_names)}")
for i, name in enumerate(eval_doc_names):
    print(f"{i:03d}\t{name}")

# (선택) DataFrame으로도 보기
pd.DataFrame({"eval_doc_index": range(len(eval_doc_names)), "doc_name": eval_doc_names})

Total EVAL_DOCS: 20
000	(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf
001	(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.pdf
002	(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.pdf
003	(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.pdf
004	2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.pdf
005	경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.pdf
006	경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.pdf
007	경기도사회서비스원_2024년 통합사회정보시스템 운영지원.pdf
008	경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).pdf
009	경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.pdf
010	국방과학연구소_대용량 자료전송시스템 고도화.pdf
011	그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.pdf
012	나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.pdf
013	남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.pdf
014	대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.pdf
015	대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.pdf
016	대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.pdf
017	대한장애인체육회_2025년 전국장애인체육대회 전산 및 시스템, 홈페이지 .pdf
018	대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .pdf
019	문화체육관광부 국립민속박물관_2024년 국립민속박물관 민속아카이브 자.pdf


,eval_doc_index,doc_name
0,0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf
1,1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...
2,2,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.pdf
3,3,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.pdf
4,4,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.pdf
5,5,경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.pdf
6,6,경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.pdf
7,7,경기도사회서비스원_2024년 통합사회정보시스템 운영지원.pdf
8,8,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).pdf
9,9,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.pdf


In [6]:
# ====== 사용자 수정 포인트: 인덱스만 바꿔서 반복 실행 ======
TARGET_DOC_INDEX = 1  # 1부터 입력: 1=첫 문서, 2=두번째 ...
# =========================================================

# 1-based -> 0-based 변환
_target_i = int(TARGET_DOC_INDEX) - 1

# 범위 체크(실수 방지)
if _target_i < 0 or _target_i >= len(EVAL_DOCS):
    raise IndexError(f"TARGET_DOC_INDEX must be 1..{len(EVAL_DOCS)} (got {TARGET_DOC_INDEX})")

target_doc_path = EVAL_DOCS[_target_i]

print("TARGET_DOC_INDEX(1-based):", TARGET_DOC_INDEX)
print("TARGET_DOC_ID(doc_id):", target_doc_path.name)
print("TARGET_DOC_PATH:", str(target_doc_path))

TARGET_DOC_INDEX(1-based): 1
TARGET_DOC_ID(doc_id): (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf
TARGET_DOC_PATH: d:\dev\github\codeit-part3-team4\data\raw\files\(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf


### fix config + build components (spec=3 C1/R2/G1)

In [7]:
CONFIG["perquestionretrieve"] = True
CONFIG["perquestiongenerate"] = True
CONFIG["savecontextperquestion"] = False

spec = ExperimentSpec(exp_id=3, chunker="C1", retriever="R2", generator="G1")
print("Running spec:", spec)

chunker, retriever, generator = make_components(spec, embed_model=embedmodel, client=client)

rag = RAGExperiment(
    chunker=chunker,
    retriever=retriever,
    generator=generator,
    questions_df=questions_df,
)

Running spec: ExperimentSpec(exp_id=3, chunker='C1', retriever='R2', generator='G1')


### run single doc

In [8]:
SIM_THRESHOLD = 80
top_k = int(CONFIG.get("top_k", CONFIG.get("topk", 15)))

doc_metrics = rag.run_single_doc_metrics(
    doc_path=target_doc_path,
    gold_fields_df=gold_fields_df,
    gold_evidence_df=gold_evidence_df,
    top_k=top_k,
    sim_threshold=SIM_THRESHOLD,
    warn_on_mismatch=True,
)

print("doc_metrics keys:", list(doc_metrics.keys())[:50])
pd.DataFrame([doc_metrics]).head(1)

doc_metrics keys: ['doc_id', 'expected_answer_count', 'answer_count', 'n_questions', 'chunk_count', 'raw_text_len', 'raw_text_preview', 'answers_preview', 'n_nonempty_answers', 'n_notfound_answers', 'pred_preview', 'n_nonempty_preds', 'n_notfound_preds', 'pred_map', 'idxs_map', 'ctx_map', 'ret_recall', 'ret_mrr', 'gen_fill', 'gen_match', 'gen_sim']


,doc_id,expected_answer_count,answer_count,n_questions,chunk_count,raw_text_len,raw_text_preview,answers_preview,n_nonempty_answers,n_notfound_answers,...,n_nonempty_preds,n_notfound_preds,pred_map,idxs_map,ctx_map,ret_recall,ret_mrr,gen_fill,gen_match,gen_sim
0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf,21,21,21,87,82,"{""performance_response_time"":""웹 페이지 조회 시 사용자가 ...","[2024년 벤처확인종합관리시스템 기능 고도화 용역사업, 벤처기업확인기관, 벤처기업...",21,2,...,21,2,{'project_name': '2024년 벤처확인종합관리시스템 기능 고도화 용역사...,"{'project_name': [47, 48, 46, 67, 56, 73, 68, ...",None,0.761905,0.48836,1.0,0.238095,44.033797


### per-question detail (gold + pred + gen metrics)

In [9]:
# sentinel은 프로젝트 버전에 따라 다를 수 있어 "둘 다" 허용
SENT_NOTFOUND_SET = {"notfound", "not_found"}   # = 정상 출력(근거 부재)
SENT_GENFAIL_SET  = {"genfail", "gen_fail"}     # = 생성 실패

EMPTY_PRED_SET = {"", "none", "null", "없음"}   # 기존 코드가 "없음"을 빈값 취급했음 [file:1]

def _norm(s: Any) -> str:
    return ("" if s is None else str(s)).strip()

def _is_notfound(s: Any) -> bool:
    return _norm(s).lower() in SENT_NOTFOUND_SET

def _is_genfail(s: Any) -> bool:
    return _norm(s).lower() in SENT_GENFAIL_SET

def _is_empty_pred(s: Any) -> bool:
    return _norm(s).lower() in EMPTY_PRED_SET

def eval_gen_pred(pred: str, gold: Optional[str], threshold: int = 80) -> Dict[str, float]:
    """
    정책(기존 호환 + 요구사항 반영):
    - fill:
      * GEN_FAIL -> 0
      * 빈문자/없음/none/null -> 0
      * 그 외 정상 텍스트 + NOT_FOUND 포함 -> 1
    - gold 비어있음 -> sim/match는 NaN, fill은 위 정책대로 유지
    - gold == NOT_FOUND:
      * pred도 NOT_FOUND -> sim=100, match=1
      * pred가 다른 텍스트 -> sim=token_set_ratio(pred, gold), match는 threshold로 결정
        (요청하신 "sim은 token_set_ratio로 계산" 반영)
    - 일반 gold -> sim=token_set_ratio, match는 threshold
    """
    pred_s = _norm(pred)
    pred_l = pred_s.lower()

    # 1) fill 계산 (NOT_FOUND는 정상 출력이므로 fill=1)
    if pred_l in SENT_GENFAIL_SET:
        fill = 0.0
    elif pred_l in EMPTY_PRED_SET:
        fill = 0.0
    else:
        fill = 1.0  # NOT_FOUND 포함

    # 2) gold 비어있으면 비교 불가 (기존과 동일하게 sim/match NaN) [file:1]
    if gold is None or _norm(gold) == "":
        return {"gen_fill": fill, "gen_match": np.nan, "gen_sim": np.nan}

    gold_s = _norm(gold)
    gold_l = gold_s.lower()

    # 3) gold가 NOT_FOUND인 경우: 별도 채점
    if gold_l in SENT_NOTFOUND_SET:
        if pred_l in SENT_NOTFOUND_SET:
            return {"gen_fill": fill, "gen_match": 1.0, "gen_sim": 100.0}
        sim = float(fuzz.token_set_ratio(pred_s, gold_s))
        return {"gen_fill": fill, "gen_match": 1.0 if sim >= threshold else 0.0, "gen_sim": sim}

    # 4) 일반 케이스: token_set_ratio로 비교
    sim = float(fuzz.token_set_ratio(pred_s, gold_s))
    match = 1.0 if sim >= threshold else 0.0
    return {"gen_fill": fill, "gen_match": match, "gen_sim": sim}


COMMON_DOC_MARK = "*"   # questions_df에서 공통 질문 표시
Q_DOC_COL = "doc_id"    # questions_df 컬럼명 확인 완료

def get_queries_for_doc(doc_id: str, questions_df: pd.DataFrame) -> List[Tuple[str, str]]:
    common = questions_df.loc[questions_df[Q_DOC_COL].astype(str) == COMMON_DOC_MARK, ["type", "question"]]
    perdoc = questions_df.loc[questions_df[Q_DOC_COL].astype(str) == doc_id, ["type", "question"]]

    merged = pd.concat([common, perdoc], ignore_index=True)
    merged["type"] = merged["type"].astype(str)
    merged["question"] = merged["question"].astype(str)

    # type 중복이면 per-doc가 common을 덮어쓰기(뒤에 있는 것을 keep)
    merged = merged.drop_duplicates(subset=["type"], keep="last")
    return list(zip(merged["type"].tolist(), merged["question"].tolist()))


doc_id = str(doc_metrics.get("doc_id", target_doc_path.name))

predmap = (
    doc_metrics.get("pred_map")
    or doc_metrics.get("predmap")
    or {}
)

queries = get_queries_for_doc(doc_id, questions_df)

gold_qdf = gold_fields_df.loc[gold_fields_df["doc_id"].astype(str) == doc_id, ["field", "gold"]].copy()
gold_qdf["field"] = gold_qdf["field"].astype(str)

rows = []
for t, q in queries:
    gold_row = gold_qdf.loc[gold_qdf["field"] == str(t)]
    gold = None if gold_row.empty else gold_row["gold"].iloc[0]

    pred = predmap.get(t, "NOT_FOUND")  # default도 기존 스타일로 통일
    g = eval_gen_pred(pred=pred, gold=gold, threshold=SIM_THRESHOLD)

    rows.append(
        {
            "doc_index": int(TARGET_DOC_INDEX),
            "doc_id": doc_id,
            "type": str(t),
            "question": str(q),
            "gold": None if gold is None else str(gold),
            "pred": None if pred is None else str(pred),
            "gen_fill": float(g["gen_fill"]) if not pd.isna(g["gen_fill"]) else np.nan,
            "gen_match": float(g["gen_match"]) if not pd.isna(g["gen_match"]) else np.nan,
            "gen_sim": float(g["gen_sim"]) if not pd.isna(g["gen_sim"]) else np.nan,
        }
    )

detail_df = pd.DataFrame(rows)

detail_view = detail_df.sort_values(
    by=["gen_match", "gen_sim", "type"],
    ascending=[True, True, True],
    na_position="last",
).reset_index(drop=True)

detail_view[["type", "question", "gold", "pred", "gen_fill", "gen_match", "gen_sim"]]

,type,question,gold,pred,gen_fill,gen_match,gen_sim
0,joint_venture_member_limit,공동수급체 구성원 수 제한은?,NOT_FOUND,5개 이하,1.0,0.0,0.000000
1,joint_venture_min_share_ratio,공동수급 시 구성원별 최소 지분율은?,NOT_FOUND,10%,1.0,0.0,0.000000
2,network_protocol_requirements,네트워크 프로토콜 지원 요구는?,"IPv4, IPv6",NOT_FOUND,1.0,0.0,0.000000
3,project_start_deadline,과업 착수 기한?,NOT_FOUND,계약일로부터 14일 이내,1.0,0.0,0.000000
4,post_contract_submission_docs,계약체결 후 10일 이내 제출해야 하는 서류는?,"['사업수행계획서(품질보증계획서, 위험관리계획서 포함)', '착수 보고서(보안확약서...",대표자용 보안확약서 및 참여자용 보안확약서,1.0,0.0,21.052632
5,budget,총 사업 예산(사업비)은 얼마인가?,"352,000,000원(부가가치세 포함)",20억원미만,1.0,0.0,21.428571
6,eval_items,평가 항목(기술/가격 등) 구성은 어떻게 되는가?,"['기술평가 90점 만점', '가격평가 10점', '전략 및 방법론 28점', '기...","기술평가와 가격평가로 구성됨. 기술평가(총점 100점, 90% 반영)는 세부 기술성...",1.0,0.0,21.958457
7,requirements_must,필수 요구사항(기능/성능/보안 등)은 무엇인가?,"['전자정부 표준프레임워크 적용', '웹표준 및 웹접근성 준수', '개인정보보호법 ...","관리적 보안(보안정책·지침 수립, 보안교육·보안서약서, 비밀유지), 물리적 보안(출...",1.0,0.0,22.051282
8,pre_deadline_required_certificates,제출 마감일 전일까지 요구되는 확인서는?,직접생산확인증명서(정보시스템개발서비스),벤처확인서,1.0,0.0,23.076923
9,purpose,사업 목적(추진 배경)은 무엇인가?,"['복수의결권주식 보고 업무처리 시스템 구축', '스톡옵션 부여·취소·철회 신고 및...","벤처기업법에 따른 복수의결권주식, 스톡옵션, 성과조건부주식 기능 고도화 및 이관, ...",1.0,0.0,26.285714


In [10]:
# 문서 1개 전체 평균 지표 계산 (per-question -> doc-level) + retrieval 지표 2개 추가
metric_cols = ["gen_fill", "gen_match", "gen_sim"]

# per-question 평균 (NaN 자동 제외)
doc_avg = detail_view[metric_cols].mean(numeric_only=True)  # NaN 자동 제외

# retrieval 지표 (doc-level)
ret_recall = float(doc_metrics.get("ret_recall", doc_metrics.get("retrecall", np.nan)))
ret_mrr    = float(doc_metrics.get("ret_mrr",    doc_metrics.get("retmrr",    np.nan)))

# 표 출력: ret_recall, ret_mrr가 위에 오도록
doc_avg_df = pd.DataFrame(
    {
        "metric": ["ret_recall", "ret_mrr"] + [f"avg_{k}" for k in doc_avg.index.tolist()],
        "avg":    [ret_recall,   ret_mrr]   + [float(v) for v in doc_avg.values.tolist()],
    }
)

print("Doc-level metrics")
display(doc_avg_df)

# dict도 같이 (로그/저장용)
doc_avg_dict = {"ret_recall": ret_recall, "ret_mrr": ret_mrr}
doc_avg_dict.update({f"avg_{k}": float(v) for k, v in doc_avg.items()})
print(doc_avg_dict)

Doc-level metrics


,metric,avg
0,ret_recall,0.761905
1,ret_mrr,0.488360
2,avg_gen_fill,1.000000
3,avg_gen_match,0.238095
4,avg_gen_sim,44.033797


{'ret_recall': 0.7619047619047619, 'ret_mrr': 0.4883597883597883, 'avg_gen_fill': 1.0, 'avg_gen_match': 0.23809523809523808, 'avg_gen_sim': 44.03379685905621}


### save single JSON

In [11]:
outdir = Path(BASE_DIR) / "outputs_single_doc"
outdir.mkdir(parents=True, exist_ok=True)

safe_doc_id = re.sub(r"[^\w\-.]+", "_", str(doc_id))
outpath = outdir / f"exp{spec.exp_id:02d}_docidx{TARGET_DOC_INDEX:02d}_{safe_doc_id}.json"

payload: Dict[str, Any] = {
    "meta": {
        "target_doc_index": int(TARGET_DOC_INDEX),
        "target_doc_id": str(doc_id),
        "target_doc_path": str(target_doc_path),
        "spec": {
            "exp_id": int(spec.exp_id),
            "chunker": spec.chunker,
            "retriever": spec.retriever,
            "generator": spec.generator,
        },
        "top_k": int(top_k),
        "sim_threshold": int(SIM_THRESHOLD),
        "perquestionretrieve": bool(CONFIG.get("perquestionretrieve", True)),
        "perquestiongenerate": bool(CONFIG.get("perquestiongenerate", True)),
        "savecontextperquestion": bool(CONFIG.get("savecontextperquestion", False)),
        "common_doc_mark": COMMON_DOC_MARK,
    },
    "doc_metrics": doc_metrics,
    "per_question": detail_view.to_dict(orient="records"),
}

with open(outpath, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", outpath)

Saved: d:\dev\github\codeit-part3-team4\outputs_single_doc\exp03_docidx01__사_벤처기업협회_2024년_벤처확인종합관리시스템_기능_고도화_용역사업_.pdf.json


### optional: only bad cases

In [12]:
tmp = detail_view.copy()
tmp["pred_l"] = tmp["pred"].fillna("").astype(str).str.strip().str.lower()

bad_cases = tmp.loc[
    (tmp["gen_match"] == 0.0)
    | (tmp["pred_l"].isin(SENT_NOTFOUND_SET | SENT_GENFAIL_SET))
].reset_index(drop=True)

print("Bad cases:", len(bad_cases))
bad_cases[["type", "question", "gold", "pred", "gen_fill", "gen_match", "gen_sim"]].head(100)

Bad cases: 17


,type,question,gold,pred,gen_fill,gen_match,gen_sim
0,joint_venture_member_limit,공동수급체 구성원 수 제한은?,NOT_FOUND,5개 이하,1.0,0.0,0.000000
1,joint_venture_min_share_ratio,공동수급 시 구성원별 최소 지분율은?,NOT_FOUND,10%,1.0,0.0,0.000000
2,network_protocol_requirements,네트워크 프로토콜 지원 요구는?,"IPv4, IPv6",NOT_FOUND,1.0,0.0,0.000000
3,project_start_deadline,과업 착수 기한?,NOT_FOUND,계약일로부터 14일 이내,1.0,0.0,0.000000
4,post_contract_submission_docs,계약체결 후 10일 이내 제출해야 하는 서류는?,"['사업수행계획서(품질보증계획서, 위험관리계획서 포함)', '착수 보고서(보안확약서...",대표자용 보안확약서 및 참여자용 보안확약서,1.0,0.0,21.052632
5,budget,총 사업 예산(사업비)은 얼마인가?,"352,000,000원(부가가치세 포함)",20억원미만,1.0,0.0,21.428571
6,eval_items,평가 항목(기술/가격 등) 구성은 어떻게 되는가?,"['기술평가 90점 만점', '가격평가 10점', '전략 및 방법론 28점', '기...","기술평가와 가격평가로 구성됨. 기술평가(총점 100점, 90% 반영)는 세부 기술성...",1.0,0.0,21.958457
7,requirements_must,필수 요구사항(기능/성능/보안 등)은 무엇인가?,"['전자정부 표준프레임워크 적용', '웹표준 및 웹접근성 준수', '개인정보보호법 ...","관리적 보안(보안정책·지침 수립, 보안교육·보안서약서, 비밀유지), 물리적 보안(출...",1.0,0.0,22.051282
8,pre_deadline_required_certificates,제출 마감일 전일까지 요구되는 확인서는?,직접생산확인증명서(정보시스템개발서비스),벤처확인서,1.0,0.0,23.076923
9,purpose,사업 목적(추진 배경)은 무엇인가?,"['복수의결권주식 보고 업무처리 시스템 구축', '스톡옵션 부여·취소·철회 신고 및...","벤처기업법에 따른 복수의결권주식, 스톡옵션, 성과조건부주식 기능 고도화 및 이관, ...",1.0,0.0,26.285714
